In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'colab'

# 1. IMPORT AND EXPLORE

products = pd.read_csv('/content/drive/MyDrive/Data Analysis/products.csv')
orders = pd.read_csv('/content/drive/MyDrive/Data Analysis/orders.csv')
marketing = pd.read_csv('/content/drive/MyDrive/Data Analysis/marketing_spend.csv')

  # 1.1 CONVERT DATES
orders['order_date'] = pd.to_datetime(orders['order_date'])

  # 1.2 CHECKING TOTAL COST CONSISTENCY
orders['calculated_total_cost'] = (
    orders['product_cost'] + orders['shipping_cost']
    + orders['platform_fee'] + orders['transaction_fee']
)

orders['cost_difference'] = orders['total_costs'] - orders['calculated_total_cost']

print("Cost mismatch summary:")
print(orders['cost_difference'].describe())

  # 1.3 CHECK IF MISMATCHES EXIST
mismatches = orders[orders['cost_difference'].abs() > 0.01]
print(f"Number of mismatched rows: {len(mismatches)}")


In [ ]:
# 2. CATEGORY PROFITABILITY

  # 2.1 GROUP BY CATEGORY

    #2.1.1 CONVERT RETURNS (YES/NO TO 1/0)
orders['returned_flag'] = orders['returned'].map({'Yes': 1, 'No': 0})

category_profit = orders.groupby('primary_category').agg(
    total_revenue=('net_revenue', 'sum'),
    gross_revenue=('gross_revenue', 'sum'),
    total_product_cost=('product_cost', 'sum'),
    total_shipping=('shipping_cost', 'sum'),
    total_fees=('platform_fee', 'sum'),
    total_discounts=('discount_amount', 'sum'),
    total_returns=('refund_amount', 'sum'),
    total_profit=('profit', 'sum'),
    total_orders=('order_id', 'count')
).reset_index()

  # 2.2 PROFIT MARGIN
category_profit['profit_margin'] = category_profit['total_profit'] / category_profit['total_revenue']

    # 2.2.1 SORT
category_profit = category_profit.sort_values('profit_margin', ascending=False)

print(category_profit)

  # 2.3 IDENTIFY TOP AND BOTTOM CATEGORIES
top_category = category_profit.head(1)
bottom_category = category_profit.tail(1)

print("Top category:")
print(top_category)

print("Bottom category:")
print(bottom_category)



In [ ]:
# 3. ORDERS CHANNEL ANALYSIS

channel_analysis = orders.groupby('channel').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('net_revenue', 'sum'),
    total_profit=('profit', 'sum'),
    avg_order_value=('net_revenue', 'mean'),
    avg_profit_per_order=('profit', 'mean'),
    total_platform_fees=('platform_fee', 'sum'),
    return_rate=('returned_flag', 'mean')
).reset_index()

  # 3.1 PROFIT MARGIN PER CHANNEL
channel_analysis['profit_margin'] = channel_analysis['total_profit'] / channel_analysis['total_revenue']

print(channel_analysis)

  # 3.2 BEST AND WORST CHANNEL
best_channel = channel_analysis.sort_values(by='avg_profit_per_order', ascending=False).head(1)
worst_channel = channel_analysis.sort_values(by='avg_profit_per_order', ascending=True).head(1)

print("Best channel:")
print(best_channel)

print("Worst channel:")
print(worst_channel)

In [ ]:
# 4. RETURN RATE AND REVENUE LOST

  # 4.1 RETURN RATE BY CATEGORY
return_by_category = orders.groupby('primary_category').agg(
    return_rate=('returned_flag', 'mean'),
    total_refunds=('refund_amount', 'sum')
).reset_index()

  # 4.2vRETURN RATE BY CHANNEL
return_by_channel = orders.groupby('channel').agg(
    return_rate=('returned_flag', 'mean'),
    total_refunds=('refund_amount', 'sum')
).reset_index()

  # 4.3 TOTAL REVENUE LOST
total_revenue_lost = orders['refund_amount'].sum()

print(return_by_category)
print(return_by_channel)
print(f"Total revenue lost to returns: {total_revenue_lost}")

In [ ]:
 # 5. MARKETING ROI ANALYSIS

  # 5.1 VERIFY ROAS
marketing['calculated_roas'] = marketing['revenue_attributed'] / marketing['spend']

  # 5.2 AGGREGATE BY PLATFORM
platform_performance = marketing.groupby('platform').agg(
    total_spend=('spend', 'sum'),
    total_revenue=('revenue_attributed', 'sum'),
    avg_roas=('roas', 'mean'),
    avg_cpc=('cpc', 'mean'),
    avg_cpa=('cpa', 'mean'),
    total_conversions=('conversions', 'sum')
).reset_index()

  # 5.3 RECALCULATE ROAS
platform_performance['overall_roas'] = (
    platform_performance['total_revenue'] / platform_performance['total_spend']
)

print(platform_performance)

  # 5.4 UNDERPERFORMING PLATFORMS
underperforming = platform_performance[platform_performance['overall_roas'] < 1]
print("Underperforming platforms:")
print(underperforming)

In [ ]:
# 6. BUDGET CUT RECOMMENDATION (20%)

  # 6.1 MONTHLY PLATFORM PERFORMANCE
monthly_performance = marketing.groupby(['month', 'platform']).agg(
    spend=('spend', 'sum'),
    revenue=('revenue_attributed', 'sum')
).reset_index()

monthly_performance['roas'] = monthly_performance['revenue'] / monthly_performance['spend']

  # 6.2 RANK WORST PERFORMERS
worst_months = monthly_performance.sort_values(by='roas').head(10)

print("Worst performing platform-month combinations:")
print(worst_months)

  # 6.3 TOTAL MARKETING SPEND
total_spend = marketing['spend'].sum()
target_cut = 0.2 * total_spend

print(f"Total spend: {total_spend}")
print(f"Target cut (20%): {target_cut}")

  # 6.4 WORST PERFORMERS UNTIL REACHING TARGET CUT
worst_months = worst_months.copy()
worst_months['cumulative_spend'] = worst_months['spend'].cumsum()

cut_plan = worst_months[worst_months['cumulative_spend'] <= target_cut]

print("Recommended cuts:")
print(cut_plan)

In [ ]:
# 7. CORE METRICS
gross_revenue = orders['gross_revenue'].sum()
net_revenue = orders['net_revenue'].sum()
total_profit = orders['profit'].sum()
total_costs = orders['total_costs'].sum()

# 8. RETURNS AND DISCOUNTS
total_returns = orders['refund_amount'].sum()
total_discounts = orders['discount_amount'].sum()

# 9. MARKET
total_marketing_spend = marketing['spend'].sum()
total_marketing_revenue = marketing['revenue_attributed'].sum()
overall_roas = total_marketing_revenue / total_marketing_spend

# 10. MARGINS
profit_margin = total_profit / net_revenue
returns_pct = total_returns / gross_revenue
discount_pct = total_discounts / gross_revenue

# 11. DATA QUALITY CHECK
orders['calc_cost_check'] = (
    orders['product_cost'] +
    orders['shipping_cost'] +
    orders['platform_fee'] +
    orders['transaction_fee']
)

valid_rows = (orders['calc_cost_check'].round(2) == orders['total_costs'].round(2)).mean()
data_quality_pct = valid_rows * 100

In [ ]:
from IPython.display import display, HTML

def card(title, value, subtitle):
    return f"""
    <div style="
        padding:15px;
        border-radius:10px;
        background-color:#111;
        color:white;
        width:220px;
        margin:10px;
        display:inline-block;">
        <h4>{title}</h4>
        <h2>{value}</h2>
        <p>{subtitle}</p>
    </div>
    """

display(HTML(
    card("Gross Revenue", f"${gross_revenue/1e3:.1f}K", "Full Year") +
    card("Net Revenue", f"${net_revenue/1e3:.1f}K", "After returns") +
    card("Total Profit", f"${total_profit/1e3:.0f}K", f"{profit_margin:.1%} margin") +
    card("Revenue Lost to Returns", f"${total_returns/1e3:.0f}K", f"{returns_pct:.1%} of gross") +
    card("Total Discount Given", f"${total_discounts/1e3:.0f}K", f"{discount_pct:.1%} of gross") +
    card("Marketing Spend", f"${total_marketing_spend/1e3:.0f}K", "Full Year") +
    card("Overall ROAS", f"{overall_roas:.1f}x", "Blended") +
    card("Data Quality", f"{data_quality_pct:.0f}%", "Verified")
))

In [ ]:
import plotly.graph_objects as go

waterfall_data = {
    "Metric": [
        "Gross Revenue",
        "Discounts",
        "Returns",
        "Net Revenue",
        "Product Cost",
        "Shipping",
        "Platform Fees",
        "Profit"
    ],
    "Value": [
        gross_revenue,
        -total_discounts,
        -total_returns,
        net_revenue,
        -orders['product_cost'].sum(),
        -orders['shipping_cost'].sum(),
        -orders['platform_fee'].sum(),
        total_profit
    ]
}

fig = go.Figure(go.Waterfall(
    x=waterfall_data["Metric"],
    y=waterfall_data["Value"],
    measure=[
        "absolute",
        "relative",
        "relative",
        "total",
        "relative",
        "relative",
        "relative",
        "total"
    ]
))

fig.update_layout(title="Profit Waterfall")
fig.show()

In [ ]:
import plotly.express as px

category = orders.groupby('primary_category').agg(
    revenue=('net_revenue', 'sum'),
    profit=('profit', 'sum')
).reset_index()

category['profit_margin'] = category['profit'] / category['revenue']

category = category.sort_values(by='profit_margin')

fig = px.bar(
    category,
    x='profit_margin',
    y='primary_category',
    orientation='h',
    title="Profit Margin by Category"
)

fig.show()

In [ ]:
returns_cat = orders.groupby('primary_category').agg(
    revenue_lost=('refund_amount', 'sum')
).reset_index()

returns_cat = returns_cat.sort_values(by='revenue_lost')

fig = px.bar(
    returns_cat,
    x='revenue_lost',
    y='primary_category',
    orientation='h',
    title="Revenue Lost to Returns by Category"
)

fig.show()

In [ ]:
category = orders.groupby('primary_category').agg(
    total_orders=('order_id', 'count'),
    revenue=('net_revenue', 'sum'),
    profit=('profit', 'sum'),
    product_cost=('product_cost', 'sum'),
    shipping_cost=('shipping_cost', 'sum'),
    platform_fee=('platform_fee', 'sum'),
    transaction_fee=('transaction_fee', 'sum'),
    returns=('refund_amount', 'sum'),
    return_rate=('returned_flag', 'mean')
).reset_index()

category['avg_profit_per_order'] = category['profit'] / category['total_orders']
category['profit_margin'] = category['profit'] / category['revenue']

In [ ]:
import plotly.express as px

fig = px.scatter(
    category,
    x='return_rate',
    y='avg_profit_per_order',
    size='revenue',
    color='primary_category',
    hover_name='primary_category',
    title="Profit per Order vs Return Rate by Category"
)

fig.show()

In [ ]:
cost_breakdown = category.melt(
    id_vars='primary_category',
    value_vars=[
        'product_cost',
        'shipping_cost',
        'platform_fee',
        'transaction_fee',
        'returns'
    ],
    var_name='cost_type',
    value_name='amount'
)

fig = px.bar(
    cost_breakdown,
    x='primary_category',
    y='amount',
    color='cost_type',
    title="Cost Breakdown by Category"
)

fig.show()

In [ ]:
category_sorted = category.sort_values(by='profit_margin')

fig = px.bar(
    category_sorted,
    x='primary_category',
    y='profit_margin',
    title="Profit Margin % by Category"
)

fig.show()

In [ ]:
channel = orders.groupby('channel').agg(
    total_orders=('order_id', 'count'),
    revenue=('net_revenue', 'sum'),
    profit=('profit', 'sum'),
    platform_fee=('platform_fee', 'sum'),
    return_rate=('returned_flag', 'mean')
).reset_index()

channel['avg_profit_per_order'] = channel['profit'] / channel['total_orders']
channel['profit_margin'] = channel['profit'] / channel['revenue']
channel['platform_fee_pct'] = channel['platform_fee'] / channel['revenue']

In [ ]:
from IPython.display import display, HTML

def channel_card(name, profit_per_order, margin, label=""):
    return f"""
    <div style="
        padding:15px;
        border-radius:10px;
        background-color:#222;
        color:white;
        width:260px;
        margin:10px;
        display:inline-block;">
        <h4>{name} {label}</h4>
        <h2>${profit_per_order:.2f}</h2>
        <p>{margin:.1%} margin</p>
    </div>
    """

# IDENTIFY BEST AND WORST
best = channel.loc[channel['avg_profit_per_order'].idxmax()]
worst = channel.loc[channel['avg_profit_per_order'].idxmin()]

cards_html = ""

for _, row in channel.iterrows():
    label = ""
    if row['channel'] == best['channel']:
        label = "— best"
    elif row['channel'] == worst['channel']:
        label = "— worst"

    cards_html += channel_card(
        row['channel'],
        row['avg_profit_per_order'],
        row['profit_margin'],
        label
    )

display(HTML(cards_html))

In [ ]:
fig = px.bar(
    channel,
    x='channel',
    y='avg_profit_per_order',
    title="Average Profit per Order by Channel"
)

fig.show()

In [ ]:
fig = px.bar(
    channel,
    x='channel',
    y=['profit_margin', 'return_rate'],
    barmode='group',
    title="Profit Margin vs Return Rate by Channel"
)

fig.show()

In [ ]:
channel_sorted = channel.sort_values(by='platform_fee_pct')

fig = px.bar(
    channel_sorted,
    x='platform_fee_pct',
    y='channel',
    orientation='h',
    title="Platform Fee as % of Revenue by Channel"
)

fig.show()

In [ ]:
platform = marketing.groupby('platform').agg(
    spend=('spend', 'sum'),
    revenue=('revenue_attributed', 'sum'),
    conversions=('conversions', 'sum'),
    avg_cpa=('cpa', 'mean')
).reset_index()

platform['roas'] = platform['revenue'] / platform['spend']
platform['cpa_calc'] = platform['spend'] / platform['conversions']

In [ ]:
top_roas = platform.sort_values(by='roas', ascending=False).head(3)
worst_roas = platform.sort_values(by='roas', ascending=True).head(1)

worst_cpa = platform.sort_values(by='cpa_calc', ascending=False).head(1)

In [ ]:
from IPython.display import display, HTML

def mkt_card(title, value, subtitle):
    return f"""
    <div style="
        padding:15px;
        border-radius:10px;
        background-color:#1a1a1a;
        color:white;
        width:300px;
        margin:10px;
        display:inline-block;">
        <h4>{title}</h4>
        <h2>{value}</h2>
        <p>{subtitle}</p>
    </div>
    """

cards_html = ""

# TOP PERFORMERS
for _, row in top_roas.iterrows():
    cards_html += mkt_card(
        "Top ROAS",
        f"{row['platform']} {row['roas']:.1f}x",
        f"${row['spend']/1e3:.1f}K spend"
    )

# WORST performer
row = worst_roas.iloc[0]
cards_html += mkt_card(
    "Worst ROAS",
    f"{row['platform']} {row['roas']:.1f}x",
    f"${row['spend']/1e3:.1f}K spend"
)

# Highest CPA
row = worst_cpa.iloc[0]
cards_html += mkt_card(
    "Highest CPA",
    f"{row['platform']} ${row['cpa_calc']:.0f}",
    "Needs attention"
)

display(HTML(cards_html))

In [ ]:
import plotly.express as px

platform_sorted = platform.sort_values(by='roas')

fig = px.bar(
    platform_sorted,
    x='platform',
    y='roas',
    title="ROAS by Platform (Full Year)"
)

fig.show()

In [ ]:
fig = px.bar(
    platform,
    x='platform',
    y=['spend', 'revenue'],
    barmode='group',
    title="Spend vs Revenue Attributed by Platform"
)

fig.show()

In [ ]:
platform_sorted_cpa = platform.sort_values(by='cpa_calc', ascending=False)

fig = px.bar(
    platform_sorted_cpa,
    x='platform',
    y='cpa_calc',
    title="Cost per Acquisition (CPA)"
)

fig.show()